# 02 — Topic Network & Keyword Evolution (NYT)

Seconda fase della pipeline: co-occurrence network delle keyword ICE, evoluzione temporale (heatmap per anno), bump chart e bubble chart per periodo.

**Input**: `data/processed/nyt/articles_ice.parquet` (generato dal notebook 01).

**Questo notebook è la pipeline canonica**, come il notebook 01: tutto il codice di `scripts/nyt/topic_network.py`, `keyword_trends.py` e `keyword_charts.py` è inline qui sotto (non importato). Eseguendolo produce gli stessi identici output di quei tre script:

- `data/processed/nyt/ice_topic_network.gexf` (grafo statico sul corpus intero)
- `data/processed/nyt/ice_topic_network_pruned.gexf` (grafo diluito: solo archi con peso ≥ soglia)
- `docs/figures/ice_edge_weight_hist.png` (distribuzione dei pesi degli archi)
- `data/processed/nyt/ice_topic_network_{periodo}.gexf` (un grafo per periodo)
- `data/processed/nyt/ice_topic_network_dynamic.gexf` (grafo dinamico, Timeline Gephi)
- `data/processed/nyt/topic_network_period_stats.csv`
- `data/processed/nyt/keyword_year_trends.csv`
- `docs/figures/keyword_trends_rate.png`, `keyword_trends_distinctiveness.png`
- `docs/figures/keyword_bump_chart.png`, `keyword_bubble_periods.png`

Una volta verificato che il notebook riproduce gli stessi output, gli script sorgente possono essere rimossi.

### Setup e Inizializzazione
In questa prima fase importiamo le librerie fondamentali per la nostra analisi dati e la teoria delle reti (come Pandas, NetworkX e Matplotlib) e impostiamo la struttura delle directory. Questo setup preparatorio è essenziale per garantire che il notebook legga correttamente il dataset pulito sull'ICE e definisca fin da subito dove verranno salvati, in modo ordinato, tutti i grafi e i grafici generati.

In [ ]:
import json
import logging
import re
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
FIGURES_DIR = BASE_DIR / "docs" / "figures"

IN_ICE = PROCESSED_DIR / "articles_ice.parquet"
OUT_GEXF = PROCESSED_DIR / "ice_topic_network.gexf"
OUT_DYNAMIC_GEXF = PROCESSED_DIR / "ice_topic_network_dynamic.gexf"
OUT_PERIOD_STATS = PROCESSED_DIR / "topic_network_period_stats.csv"
OUT_KEYWORD_TRENDS_CSV = PROCESSED_DIR / "keyword_year_trends.csv"
OUT_RATE_PNG = FIGURES_DIR / "keyword_trends_rate.png"
OUT_DISTINCT_PNG = FIGURES_DIR / "keyword_trends_distinctiveness.png"
OUT_BUMP_PNG = FIGURES_DIR / "keyword_bump_chart.png"
OUT_BUBBLE_PNG = FIGURES_DIR / "keyword_bubble_periods.png"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("topic_network")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")


### Caricamento e Pulizia delle Keyword
In questo passaggio carichiamo il dataset degli articoli, estraendo gli identificativi, le date di pubblicazione e decodificando il campo delle keyword dal formato JSON. Parallelamente, definiamo la logica per filtrare le parole chiave: conserviamo solo le categorie editorialmente rilevanti (soggetti, organizzazioni, luoghi, persone) e rimuoviamo esplicitamente la stringa "immigration and customs enforcement (us)". Poiché l'intero corpus è già focalizzato sull'ICE, mantenere questa parola chiave genererebbe un nodo gigantesco che finirebbe per dominare il grafo e oscurare i veri collegamenti tematici della narrazione.

Grafo di co-occorrenza: nodi = keyword NYT associate agli articoli ICE, archi = quante volte due keyword compaiono nello stesso articolo. Il valore ICE stesso ("Immigration and Customs Enforcement (US)") è escluso dai nodi — presente per definizione in ogni articolo del dataset, diventerebbe un hub universale non informativo.

**Pulizia delle keyword (2026-10-05).** `extract_keywords` applica due correzioni emerse nel notebook 10:

1. **Duplicati nello stesso articolo.** In 2 articoli la stessa keyword compare due volte nella lista del NYT (`sandoval-perez, josue noe`; `deportation`, `justice department`, `sessions, jefferson b iii`). Senza deduplicare, il loro `count` risultava gonfiato di 1 e nascevano 4 archi di una keyword con se stessa (*self-loop*). Si tiene solo la prima occorrenza.
2. **Tag tecnici del NYT.** Dal 2025 tra le keyword `subject` compaiono etichette del sistema editoriale, non temi: `internal-open-access-from-nl`, `content type: personal profile`, `live-detached`, `internal-storyline-no` e altre (14), più il tono dell'audio (`audio-neutral-informative`, `audio-negative-sad`..., 5), il tipo di contenuto visivo (`vis-photo`, `vis-video`, `vis-design`) e i feed della app (`your-feed-science`..., 6): 28 tag in tutto. Sono esclusi con l'espressione regolare `TECHNICAL_TAG_RE`.


In [ ]:
def load_ice_articles() -> pd.DataFrame:
    """Carica articles_ice.parquet (colonne: id, pub_date, keywords_json, ...)."""
    df = pd.read_parquet(IN_ICE, columns=["id", "pub_date", "keywords_json"])
    df["keywords"] = df["keywords_json"].apply(json.loads)
    return df[["id", "pub_date", "keywords"]]


# tag del sistema editoriale NYT finiti tra le keyword `subject` dal 2025 (es. "internal-open-access-from-nl", "audio-neutral-informative")
TECHNICAL_TAG_RE = re.compile(r"^(internal-|content type:|live-detached$|audio-|vis-|your-feed-)")


def extract_keywords(keywords: list) -> list[tuple[str, int, str]]:
    """Da una lista di dict a lista di keyword (value, rank, category) pulite di UN SOLO articolo per il grafo.

    Senza duplicati: se una keyword compare due volte nello stesso articolo si tiene la prima occorrenza.
    """
    NAME_ALIASES = {
        "subject": "subject",
        "organizations": "organizations",
        "organization": "organizations",
        "persons": "persons",
        "person": "persons",
        "glocations": "glocations",
        "location": "glocations",
        "creative_works": "creative_works",
        "title": "creative_works",
    }
    included_names = {"subject", "persons", "glocations", "organizations", "creative_works"}

    ice_string = "immigration and customs enforcement (us)"
    keywords_filtered = []
    seen = set()

    for kw in keywords or []:
        raw_name = kw.get("name", "").strip().lower()
        name = NAME_ALIASES.get(raw_name)

        if name in included_names:
            value = kw.get("value", "").strip().lower()
            if value and value != ice_string and value not in seen and not TECHNICAL_TAG_RE.match(value):
                seen.add(value)
                rank = kw.get("rank", 0)
                keywords_filtered.append((value, rank, name))

    return keywords_filtered


df = load_ice_articles()
print(f"Articoli ICE caricati: {len(df):,}")


In [ ]:
# Demo: keyword estratte/normalizzate dal primo articolo del dataset
sample_row = df.iloc[0]
print(f"Articolo: {sample_row['id']}")
extract_keywords(sample_row["keywords"])


### Costruzione del Grafo di Co-occorrenza
 Attraversando ogni articolo, la funzione genera un collegamento (arco pesato) ogni volta che due keyword compaiono insieme, rafforzando l'intensità del legame se la coppia si ripete nel corpus. Arricchiamo inoltre i singoli nodi con metadati cruciali: calcoliamo l'importanza media (rank) della parola e le assegniamo la categoria editoriale predominante (un passaggio tecnico essenziale, dato che il formato di esportazione non digerisce strutture dati annidate). Infine, esportiamo la rete completa in formato GEXF, pronta per essere esplorata su software come Gephi per mappare visivamente l'architettura complessiva della narrazione.

In [ ]:
def build_cooccurrence_graph(df: pd.DataFrame) -> nx.Graph:
    """Costruisce il grafo di co-occorrenza: un arco per ogni coppia di keyword nello stesso articolo."""
    graph = nx.Graph()

    for _, row in df.iterrows():
        keywords = extract_keywords(row["keywords"])

        for val, rank, category in keywords:
            if graph.has_node(val):
                graph.nodes[val]["count"] += 1
                graph.nodes[val]["sum_rank"] += rank
                graph.nodes[val]["category_counts"][category] = (
                    graph.nodes[val]["category_counts"].get(category, 0) + 1
                )
            else:
                graph.add_node(val, count=1, sum_rank=rank, category_counts={category: 1})

        if len(keywords) < 2:
            continue

        keywords_values = [kw[0] for kw in keywords]
        edges = combinations(keywords_values, 2)

        for u, v in edges:
            if graph.has_edge(u, v):
                graph[u][v]["weight"] += 1
            else:
                graph.add_edge(u, v, weight=1)

    # category_counts (dict annidato) non e' esportabile in GEXF: lo riduciamo a
    # 'category' (valore singolo, la categoria piu' frequente per quel nodo).
    for node, data in graph.nodes(data=True):
        data["avg_rank"] = data["sum_rank"] / data["count"] if data["count"] > 0 else 0
        category_counts = data.pop("category_counts")
        data["category"] = max(category_counts, key=category_counts.get)

    return graph


def export_graph(graph: nx.Graph, path: Path) -> None:
    """Scrive il grafo in formato GEXF (path)."""
    nx.write_gexf(graph, path)


log.info("Costruisco co-occurrence network su %d articoli (corpus intero)", len(df))
graph = build_cooccurrence_graph(df)
log.info("Grafo intero: %d nodi, %d archi", graph.number_of_nodes(), graph.number_of_edges())
export_graph(graph, OUT_GEXF)
log.info("Scritto %s", OUT_GEXF.relative_to(BASE_DIR))


In [ ]:
# Sanity check: i nodi piu' frequenti devono essere temi coerenti (nessun rumore)
top_nodes = sorted(graph.nodes(data=True), key=lambda x: -x[1]["count"])[:15]
pd.DataFrame(
    [{"keyword": n, "count": d["count"], "avg_rank": round(d["avg_rank"], 2), "category": d["category"]} for n, d in top_nodes]
)


### Grafo diluito: soglia sui pesi degli archi
Il grafo completo è troppo denso per essere letto (2.441 nodi, 36.654 archi): la maggior parte degli archi ha peso 1, cioè due keyword comparse insieme in un solo articolo. Qui guardiamo prima la distribuzione dei pesi (istogramma in scala logaritmica, più una tabella che mostra cosa resta a ogni soglia), poi costruiamo una versione diluita del grafo tenendo solo gli archi con peso ≥ `MIN_EDGE_WEIGHT` ed eliminando i nodi rimasti senza archi.

Il grafo diluito parte dal grafo standard `graph` (non viene ricostruito dagli articoli): gli attributi dei nodi (`count`, `avg_rank`, `category`) restano quelli calcolati sul corpus intero. La soglia è una scelta, non un valore ricavato dai dati: la distribuzione è a coda lunga e non ha un punto di taglio naturale, quindi va dichiarata nel report.

In [ ]:
OUT_WEIGHT_HIST_PNG = FIGURES_DIR / "ice_edge_weight_hist.png"
OUT_PRUNED_GEXF = PROCESSED_DIR / "ice_topic_network_pruned.gexf"
MIN_EDGE_WEIGHT = 5


def prune_graph(graph: nx.Graph, min_weight: int) -> nx.Graph:
    """Copia del grafo con i soli archi di peso >= min_weight; i nodi rimasti isolati sono rimossi."""
    pruned = graph.copy()
    pruned.remove_edges_from([(u, v) for u, v, w in graph.edges(data="weight") if w < min_weight])
    pruned.remove_nodes_from(list(nx.isolates(pruned)))
    return pruned


def threshold_table(graph: nx.Graph, thresholds: list[int]) -> pd.DataFrame:
    """Per ogni soglia: nodi/archi rimasti, componente gigante e quota del peso totale conservata."""
    total_weight = sum(w for _, _, w in graph.edges(data="weight"))
    rows = []
    for t in thresholds:
        pruned = prune_graph(graph, t)
        giant = max((len(c) for c in nx.connected_components(pruned)), default=0)
        rows.append({
            "soglia": t,
            "nodi": pruned.number_of_nodes(),
            "archi": pruned.number_of_edges(),
            "componenti": nx.number_connected_components(pruned),
            "nodi_componente_gigante": giant,
            "peso_conservato_pct": round(sum(w for _, _, w in pruned.edges(data="weight")) / total_weight * 100, 1),
        })
    return pd.DataFrame(rows)


def plot_weight_histogram(graph: nx.Graph, min_weight: int, out_path: Path) -> None:
    weights = np.array([w for _, _, w in graph.edges(data="weight")])
    # bin logaritmici: i pesi vanno da 1 a ~700 e il 75% vale 1, bin lineari mostrerebbero una sola barra
    bins = np.unique(np.round(np.logspace(0, np.log10(weights.max() + 1), 30))) - 0.5

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(weights, bins=bins, color="#1f77b4", edgecolor="white", linewidth=0.5)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(min_weight - 0.5, color="#d62728", linestyle="--", linewidth=1.5, label=f"soglia = {min_weight}")
    ax.set_xlabel("Peso dell'arco (articoli in cui le due keyword compaiono insieme)")
    ax.set_ylabel("Numero di archi")
    ax.set_title("NYT — Distribuzione dei pesi degli archi nel grafo di co-occorrenza ICE")
    ax.legend()
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


plot_weight_histogram(graph, MIN_EDGE_WEIGHT, OUT_WEIGHT_HIST_PNG)
log.info("Scritto %s", OUT_WEIGHT_HIST_PNG.relative_to(BASE_DIR))
display(Image(filename=str(OUT_WEIGHT_HIST_PNG)))
threshold_table(graph, [1, 2, 3, 4, 5, 8, 10, 15, 20, 30, 50])

In [ ]:
pruned_graph = prune_graph(graph, MIN_EDGE_WEIGHT)
log.info(
    "Grafo diluito (peso >= %d): %d nodi, %d archi (da %d nodi, %d archi)",
    MIN_EDGE_WEIGHT, pruned_graph.number_of_nodes(), pruned_graph.number_of_edges(),
    graph.number_of_nodes(), graph.number_of_edges(),
)
export_graph(pruned_graph, OUT_PRUNED_GEXF)
log.info("Scritto %s", OUT_PRUNED_GEXF.relative_to(BASE_DIR))

# Archi piu' pesanti: devono essere coppie di temi sensate
top_edges = sorted(pruned_graph.edges(data="weight"), key=lambda e: (-e[2], e[0], e[1]))[:15]
pd.DataFrame(top_edges, columns=["keyword_a", "keyword_b", "weight"])

### Analisi Longitudinale e Slicing Temporale
In questo blocco introduciamo la dimensione temporale nella nostra analisi di rete. Suddividiamo l'intero corpus in quattro ere storiche distinte (che ricalcano i mandati presidenziali) e generiamo un grafo di co-occorrenza separato per ciascun periodo. Per ogni sotto-rete estraiamo subito le metriche topologiche globali — come la densità e il grado pesato medio — esportando poi i risultati in formato CSV e le singole reti in GEXF.

In [ ]:
def get_periods(df: pd.DataFrame) -> list[tuple[str, int, int]]:
    """Confini dei periodi da confrontare: (label, start_year incluso, end_year escluso)."""
    max_year = int(df["pub_date"].dt.year.max())
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def filter_by_period(df: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    """Filtra df per anno di pubblicazione in [start_year, end_year)."""
    years = df["pub_date"].dt.year
    return df[(years >= start_year) & (years < end_year)]


def graph_stats(graph: nx.Graph) -> dict:
    """Metriche di confronto rapido tra grafi di periodi diversi."""
    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()
    density = nx.density(graph) if n_nodes > 1 else 0.0
    avg_weighted_degree = (
        sum(w for _, _, w in graph.edges(data="weight")) * 2 / n_nodes if n_nodes else 0.0
    )
    return {"nodes": n_nodes, "edges": n_edges, "density": density, "avg_weighted_degree": avg_weighted_degree}


def build_period_graphs(df: pd.DataFrame, periods: list[tuple[str, int, int]]) -> dict:
    """Costruisce un grafo di co-occorrenza per ciascun periodo. Ritorna {label: (graph, start, end, n_articoli)}."""
    period_graphs = {}
    for label, start, end in periods:
        period_df = filter_by_period(df, start, end)
        pgraph = build_cooccurrence_graph(period_df)
        period_graphs[label] = (pgraph, start, end, len(period_df))
        log.info(
            "Periodo %s (%d articoli): %d nodi, %d archi",
            label, len(period_df), pgraph.number_of_nodes(), pgraph.number_of_edges(),
        )
    return period_graphs


def export_period_stats(period_graphs: dict, path: Path) -> None:
    """CSV di confronto (nodi/archi/densita'/grado pesato medio) tra periodi."""
    rows = []
    for label, (pgraph, start, end, n_articles) in period_graphs.items():
        rows.append({
            "period": label, "start_year": start, "end_year_excl": end,
            "n_articles": n_articles, **graph_stats(pgraph),
        })
    pd.DataFrame(rows).to_csv(path, index=False)


periods = get_periods(df)
period_graphs = build_period_graphs(df, periods)

export_period_stats(period_graphs, OUT_PERIOD_STATS)
log.info("Scritto %s", OUT_PERIOD_STATS.relative_to(BASE_DIR))

for label, (period_graph, *_rest) in period_graphs.items():
    out_path = PROCESSED_DIR / f"ice_topic_network_{label}.gexf"
    export_graph(period_graph, out_path)
    log.info("Scritto %s", out_path.relative_to(BASE_DIR))

pd.read_csv(OUT_PERIOD_STATS)


### Creazione del Grafo Dinamico (Timeline)
In questa fase consolidiamo le reti delle singole ere storiche in un'unica architettura dinamica. Assegnando marcatori temporali specifici (definiti "spells") a ogni nodo e a ogni arco, registriamo gli esatti intervalli di anni in cui le keyword e le loro connessioni sono attive. Questo ci permette di generare un file GEXF avanzato, progettato per essere letto dalla funzione Timeline di Gephi. Il vantaggio analitico è notevole: invece di confrontare grafi statici separati, potremo animare l'evoluzione della rete e osservare in modo fluido come la narrazione giornalistica sull'ICE muta, si polarizza o si disgrega col passare del tempo.

In [ ]:
def build_dynamic_graph(period_graphs: dict) -> nx.Graph:
    """Unisce i grafi per periodo in un unico grafo dinamico per la Timeline di Gephi."""
    merged = nx.Graph()
    merged.graph["mode"] = "dynamic"

    for label, (pgraph, start, end, _n_articles) in period_graphs.items():
        start, end = float(start), float(end)
        for node, data in pgraph.nodes(data=True):
            if not merged.has_node(node):
                merged.add_node(node, count=0, spells=[], count_by_period=[])
            merged.nodes[node]["count"] += data["count"]
            merged.nodes[node]["spells"].append((start, end))
            merged.nodes[node]["count_by_period"].append((data["count"], start, end))

        for u, v, edata in pgraph.edges(data=True):
            if not merged.has_edge(u, v):
                merged.add_edge(u, v, weight=0, spells=[], weight_by_period=[])
            merged[u][v]["weight"] += edata["weight"]
            merged[u][v]["spells"].append((start, end))
            merged[u][v]["weight_by_period"].append((edata["weight"], start, end))

    return merged


dynamic_graph = build_dynamic_graph(period_graphs)
export_graph(dynamic_graph, OUT_DYNAMIC_GEXF)
log.info(
    "Scritto %s (%d nodi, %d archi, mode=dynamic)",
    OUT_DYNAMIC_GEXF.relative_to(BASE_DIR), dynamic_graph.number_of_nodes(), dynamic_graph.number_of_edges(),
)


### Estrazione delle Tendenze Temporali e Distintività
In questa fase calcoliamo le metriche semantiche anno per anno, passando da una logica di rete a un'analisi delle serie temporali. Il codice genera una tabella dettagliata che confronta la frequenza locale di una keyword in un determinato anno con la sua frequenza globale sull'intero corpus. Questo ci permette di calcolare la *Distinctiveness* (distintività): filtra il "rumore di fondo" delle parole sempre presenti (come "confine" o "polizia") per far emergere i termini specifici che hanno caratterizzato in modo univoco una precisa annata.

- **rate_pct**: keyword più frequenti per anno, normalizzate sul volume ICE di quell'anno (confrontabile tra 2004 e 2025).
- **distinctiveness**: `log2(rate dell'anno / rate medio storico della keyword)` — fa emergere cosa è *caratteristico di quell'anno* (es. "family separation" nel 2018) invece delle keyword sempre in cima (Trump, deportation...) che dominerebbero ogni anno nella classifica per rate grezzo.

Soglie minime (`MIN_COUNT_YEAR=3`, `MIN_TOTAL_GLOBAL=8`) tagliano il rumore di keyword con 1-2 occorrenze che altrimenti entrerebbero in classifica con valori enormi ma non significativi.

In [ ]:
MIN_COUNT_YEAR = 3
MIN_TOTAL_GLOBAL = 8
TOP_N_PER_YEAR = 8


def build_year_keyword_table(df: pd.DataFrame) -> pd.DataFrame:
    """Tabella lunga (year, keyword, count, n_articles, rate_pct, total_count, global_rate_pct, distinctiveness)."""
    records = []
    for row in df.itertuples():
        year = row.pub_date.year
        values = {value for value, _rank, _category in extract_keywords(row.keywords)}
        records.extend((year, value) for value in values)

    long_df = pd.DataFrame(records, columns=["year", "keyword"])
    counts = long_df.groupby(["year", "keyword"]).size().rename("count").reset_index()

    year_totals = df["pub_date"].dt.year.value_counts().rename("n_articles")
    counts = counts.merge(year_totals, left_on="year", right_index=True)
    counts["rate_pct"] = counts["count"] / counts["n_articles"] * 100

    total_articles = int(year_totals.sum())
    global_counts = counts.groupby("keyword")["count"].sum().rename("total_count").reset_index()
    global_counts["global_rate_pct"] = global_counts["total_count"] / total_articles * 100
    counts = counts.merge(global_counts, on="keyword")
    counts["distinctiveness"] = counts["rate_pct"] / counts["global_rate_pct"]

    return counts.sort_values(["year", "count"], ascending=[True, False]).reset_index(drop=True)


def select_top_rows(counts, score_col, min_count_year=MIN_COUNT_YEAR, min_total_global=0, top_n=TOP_N_PER_YEAR):
    """Unione delle top-N keyword per anno secondo score_col, sopra le soglie minime."""
    eligible = counts[(counts["count"] >= min_count_year) & (counts["total_count"] >= min_total_global)]
    top = eligible.sort_values(["year", score_col], ascending=[True, False]).groupby("year").head(top_n)
    order = (
        top.groupby("keyword")
        .agg(first_year=("year", "min"), max_score=(score_col, "max"))
        .sort_values(["first_year", "max_score"], ascending=[True, False])
    )
    return order.index.tolist()


log.info("Costruisco tabella year x keyword su %d articoli", len(df))
counts = build_year_keyword_table(df)
counts.to_csv(OUT_KEYWORD_TRENDS_CSV, index=False)
log.info("Scritto %s (%d righe)", OUT_KEYWORD_TRENDS_CSV.relative_to(BASE_DIR), len(counts))
counts.head()


In [ ]:
def plot_heatmap(counts, keywords, value_col, out_path, title, cbar_label, cmap, center=None):
    years = sorted(counts["year"].unique())
    pivot = counts.pivot(index="keyword", columns="year", values=value_col).reindex(index=keywords, columns=years)

    mask_missing = center is not None
    data = pivot.to_numpy(dtype=float)
    if not mask_missing:
        data = np.nan_to_num(data, nan=0.0)

    # altezza compatta: 0.16in/keyword invece di 0.35in, dpi 120 invece di 150 —
    # stessa informazione (tutte le keyword/anni), immagine molto piu' piccola
    fig_h = max(3.5, 0.16 * len(keywords) + 1.2)
    fig, ax = plt.subplots(figsize=(10, fig_h))

    cmap_obj = plt.get_cmap(cmap).copy()
    if mask_missing:
        cmap_obj.set_bad("#dddddd")
        vmax = np.nanmax(np.abs(data)) if np.isfinite(data).any() else 1.0
        im = ax.imshow(np.ma.masked_invalid(data), cmap=cmap_obj, vmin=-vmax, vmax=vmax, aspect="auto")
    else:
        im = ax.imshow(data, cmap=cmap_obj, vmin=0, aspect="auto")

    ax.set_xticks(range(len(years)))
    ax.set_xticklabels(years, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(keywords)))
    ax.set_yticklabels(keywords, fontsize=6)
    ax.set_title(title, fontsize=10)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label, fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


rate_keywords = select_top_rows(counts, "rate_pct")
plot_heatmap(
    counts, rate_keywords, "rate_pct", OUT_RATE_PNG,
    title="NYT — Top keyword ICE per anno (rate = % articoli ICE dell'anno)",
    cbar_label="% articoli ICE dell'anno", cmap="viridis",
)
log.info("Scritto %s (%d keyword)", OUT_RATE_PNG.relative_to(BASE_DIR), len(rate_keywords))
display(Image(filename=str(OUT_RATE_PNG)))

In [ ]:
counts["log2_distinctiveness"] = np.log2(counts["distinctiveness"])
distinct_keywords = select_top_rows(counts, "distinctiveness", min_total_global=MIN_TOTAL_GLOBAL)
plot_heatmap(
    counts, distinct_keywords, "log2_distinctiveness", OUT_DISTINCT_PNG,
    title="NYT — Keyword più distintive per anno (log2 vs rate medio storico della keyword)",
    cbar_label="log2(rate anno / rate medio storico keyword)", cmap="RdBu_r", center=0.0,
)
log.info("Scritto %s (%d keyword)", OUT_DISTINCT_PNG.relative_to(BASE_DIR), len(distinct_keywords))
display(Image(filename=str(OUT_DISTINCT_PNG)))


## 3. Bump chart e bubble chart

- **Bump chart**: rank annuale (1 = keyword più frequente) delle keyword ICE principali — il rank è calcolato solo tra le keyword selezionate (non tra tutte le migliaia dell'anno), per mostrare l'ordine relativo nel piccolo gruppo scelto invece di schiacciare le linee in cima al grafico.
- **Bubble chart**: uno per periodo, x = prominenza NYT (avg_rank, invertito), y = frequenza normalizzata, size = grado nel grafo di co-occorrenza del periodo, colore = categoria NYT.

In [ ]:
CATEGORY_COLORS = {
    "subject": "#1f77b4",
    "persons": "#d62728",
    "glocations": "#2ca02c",
    "organizations": "#9467bd",
    "creative_works": "#8c564b",
}
CATEGORY_LABELS_IT = {
    "subject": "tema/soggetto",
    "persons": "persona",
    "glocations": "luogo",
    "organizations": "organizzazione",
    "creative_works": "opera/titolo",
}
TOP_N_BUMP = 6
TOP_N_BUBBLE = 25
TOP_N_LABELS = 12


def plot_bump_chart(df: pd.DataFrame, out_path: Path) -> None:
    counts_local = build_year_keyword_table(df)
    selected = select_top_rows(counts_local, "count", top_n=TOP_N_BUMP)

    sub = counts_local[counts_local["keyword"].isin(selected)].copy()
    sub["rank"] = sub.groupby("year")["count"].rank(method="first", ascending=False)

    years = sorted(counts_local["year"].unique())
    pivot = sub.pivot(index="keyword", columns="year", values="rank").reindex(index=selected, columns=years)

    fig, ax = plt.subplots(figsize=(13, 7))
    cmap = plt.get_cmap("tab20")

    for i, keyword in enumerate(selected):
        ranks = pivot.loc[keyword]
        color = cmap(i % 20)
        ax.plot(years, ranks, marker="o", linewidth=1.8, markersize=4, color=color)

        valid = ranks.dropna()
        if not valid.empty:
            last_year = valid.index[-1]
            ax.annotate(
                keyword, (last_year, valid.iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=8, color=color,
            )

    ax.invert_yaxis()
    ax.set_xlabel("Anno")
    ax.set_ylabel("Rank annuale (1 = keyword più frequente dell'anno)")
    ax.set_title("NYT — Evoluzione del rank delle keyword ICE principali")
    ax.set_xticks(years)
    ax.set_xticklabels(years, rotation=45, ha="right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


plot_bump_chart(df, OUT_BUMP_PNG)
log.info("Scritto %s", OUT_BUMP_PNG.relative_to(BASE_DIR))
display(Image(filename=str(OUT_BUMP_PNG)))


### Visualizzazione Multidimensionale per Ere Storiche
In questa fase finale generiamo un grafico a bolle (Bubble Chart) a quattro quadranti per confrontare visivamente i diversi periodi storici. Questa visualizzazione è particolarmente potente perché incrocia simultaneamente quattro variabili cruciali per ogni parola chiave: la frequenza di utilizzo (sull'asse verticale), la prominenza editoriale dettata dal NYT (sull'asse orizzontale, basata sull'ordine di apparizione), la connettività tematica (la dimensione della bolla, che riflette il numero di collegamenti nella rete) e la natura semantica (il colore associato alla categoria). Creando questa griglia comparativa possiamo sintetizzare in una singola immagine l'intera evoluzione della narrazione sull'ICE, osservando a colpo d'occhio come i temi principali si siano spostati dalle retrovie ai titoli di testa, o come abbiano acquisito o perso centralità al variare dei mandati presidenziali.

In [ ]:
def period_bubble_frame(graph, n_articles: int) -> pd.DataFrame:
    rows = []
    for node, data in graph.nodes(data=True):
        rows.append({
            "keyword": node,
            "rate_pct": data["count"] / n_articles * 100 if n_articles else 0.0,
            "avg_rank": data["avg_rank"],
            "degree": graph.degree(node),
            "category": data.get("category", "subject"),
        })
    return pd.DataFrame(rows)


def plot_bubble_chart(period_graphs: dict, out_path: Path) -> None:
    tops = {}
    for label, (pgraph, _start, _end, n_articles) in period_graphs.items():
        frame = period_bubble_frame(pgraph, n_articles)
        tops[label] = (frame.sort_values("rate_pct", ascending=False).head(TOP_N_BUBBLE), n_articles)

    all_top = pd.concat(f for f, _n in tops.values())
    x_max = all_top["avg_rank"].max() * 1.05
    y_max = all_top["rate_pct"].max() * 1.1
    degree_max = max(all_top["degree"].max(), 1)

    fig, axes = plt.subplots(2, 2, figsize=(15, 12), sharex=True, sharey=True)

    for ax, (label, (top, n_articles)) in zip(axes.flat, tops.items()):
        colors = top["category"].map(CATEGORY_COLORS).fillna("#7f7f7f")
        sizes = 25 + 475 * (top["degree"] / degree_max)
        ax.scatter(top["avg_rank"], top["rate_pct"], s=sizes, c=colors, alpha=0.65, edgecolors="black", linewidths=0.4)
        for _, row in top.head(TOP_N_LABELS).iterrows():
            ax.annotate(row["keyword"], (row["avg_rank"], row["rate_pct"]), fontsize=7, xytext=(4, 2), textcoords="offset points")
        ax.set_title(f"{label}  (n={n_articles} articoli ICE)")
        ax.set_xlim(0, x_max)
        ax.set_ylim(0, y_max)
        ax.invert_xaxis()
        ax.grid(alpha=0.3)

    fig.supxlabel("Prominenza NYT (avg rank keyword nell'articolo — verso destra = più prominente)")
    fig.supylabel("% articoli ICE del periodo con la keyword")
    fig.suptitle(
        "NYT — Keyword ICE per periodo: prominenza vs frequenza vs connettività tematica "
        "(dimensione bubble = grado nella rete di co-occorrenza)",
        fontsize=12,
    )

    legend_handles = [
        mlines.Line2D([], [], color=color, marker="o", linestyle="", markersize=8, label=CATEGORY_LABELS_IT[cat])
        for cat, color in CATEGORY_COLORS.items()
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.02))

    fig.tight_layout(rect=(0.02, 0.03, 1, 0.95))
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


plot_bubble_chart(period_graphs, OUT_BUBBLE_PNG)
log.info("Scritto %s", OUT_BUBBLE_PNG.relative_to(BASE_DIR))
display(Image(filename=str(OUT_BUBBLE_PNG)))


## Riepilogo

Output scritti da questo notebook (stessi path/nomi già prodotti da `topic_network.py`, `keyword_trends.py`, `keyword_charts.py`):

- `ice_topic_network.gexf`, `ice_topic_network_pruned.gexf`, `ice_topic_network_{2003-2016,2017-2020,2021-2024,2025-presente}.gexf`, `ice_topic_network_dynamic.gexf`
- `topic_network_period_stats.csv`, `keyword_year_trends.csv`
- `ice_edge_weight_hist.png`, `keyword_trends_rate.png`, `keyword_trends_distinctiveness.png`, `keyword_bump_chart.png`, `keyword_bubble_periods.png`

